# Paper 2 — EMF-SLT: Explicit Multi-Modal Fusion (ICASSP 2024)
**Hu et al.** — *An Explicit Multi-Modal Fusion Method for Sign Language Translation*

### Architecture (Paper §2, Fig.2):
```
ResNet18 features (512D)
  → Sign Encoder (Transformer)
  + Gloss Encoder (Transformer)
  → Vector Quantizer  [§2.1, Eq.1-2]  aligns sign↔gloss in shared codebook
  → Fusion Module     [§2.1, Eq.3-5]  cross-attention [S;S'] × S
  → S2T Decoder + G2T Decoder         [§2.2, Eq.6]
  Multi-task: L = NLL + α·KL + β·Align  [Eq.8, α=1, β=0.5]
```
### Evaluation (Paper §3.1): BLEU-4 + ROUGE-L
### Training: **loss ↓ + token accuracy % ↑** shown live every epoch

In [ ]:
import os, re, json, math, random
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'text.color':'#e6edf3','axes.titlecolor':'#58a6ff',
    'xtick.color':'#8b949e','grid.color':'#21262d','font.size':11})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE   = r'D:\ISL2\ISL_CSLRT_Corpus'
FRAMES = os.path.join(BASE, 'Frames_Sentence_Level')
FEATS  = os.path.join(BASE, 'resnet18_features')   # 512D ResNet18 features
CKPT   = os.path.join(BASE, 'best_paper2_emfslt.pth')
PAD, BOS, EOS = '<pad>', '<bos>', '<eos>'
TR, VA, TE = ['1','2','3','4','5'], ['6'], ['7']
ALPHA, BETA, GAMMA = 1.0, 0.5, 0.1   # Paper 2 Eq(8): α=1, β=0.5; +CTC

torch.manual_seed(42); np.random.seed(42); random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda': print(torch.cuda.get_device_name(0))

## Step 1 — Extract/Cache ResNet18 Features
> 512D visual features per frame. Skips if already cached.

In [ ]:
from tqdm import tqdm
from torchvision import models, transforms

def tokenize(t):
    if not t: return []
    return [w for w in re.sub(r"[^a-z0-9 ']", ' ', str(t).lower().strip()).split() if w]

f2s = {f: ' '.join(tokenize(f)) for f in sorted(os.listdir(FRAMES))}

resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
fext   = nn.Sequential(*list(resnet.children())[:-1]).to(DEVICE).eval()
for p in fext.parameters(): p.requires_grad = False
prep   = transforms.Compose([
    transforms.Resize((112,112)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

exist = sum(len([f for f in os.listdir(os.path.join(FEATS,sp)) if f.endswith('.npy')])
    for sp in ['train','val','test'] if os.path.isdir(os.path.join(FEATS,sp)))

if exist > 100:
    print(f'✅ ResNet18 features cached ({exist} files). Skipping.')
else:
    print('Extracting ResNet18 features...')
    for sp in ['train','val','test']: os.makedirs(os.path.join(FEATS,sp), exist_ok=True)
    done = 0
    for sf in tqdm(sorted(os.listdir(FRAMES))):
        for sig in sorted(os.listdir(os.path.join(FRAMES,sf))):
            sigp = os.path.join(FRAMES, sf, sig)
            if not os.path.isdir(sigp): continue
            split = 'train' if sig in TR else ('val' if sig in VA else 'test')
            safe  = re.sub(r'[^\w]','_',f'{sf}_s{sig}')[:100]
            no = os.path.join(FEATS, split, f'{safe}.npy')
            lo = os.path.join(FEATS, split, f'{safe}_label.txt')
            if os.path.exists(no): done += 1; continue
            imgs = [f for f in sorted(os.listdir(sigp)) if f.lower().endswith(('.jpg','.png'))]
            feats, batch = [], []
            for img in imgs:
                try: batch.append(prep(Image.open(os.path.join(sigp,img)).convert('RGB')))
                except: continue
                if len(batch) == 32:
                    with torch.no_grad():
                        feats.append(fext(torch.stack(batch).to(DEVICE)).squeeze(-1).squeeze(-1).cpu().numpy())
                    batch = []
            if batch:
                with torch.no_grad():
                    feats.append(fext(torch.stack(batch).to(DEVICE)).squeeze(-1).squeeze(-1).cpu().numpy())
            if not feats: continue
            np.save(no, np.concatenate(feats, 0))
            open(lo, 'w', encoding='utf-8').write(f2s[sf])
            done += 1
    print(f'Done: {done}')

for sp in ['train','val','test']:
    d = os.path.join(FEATS, sp)
    if os.path.isdir(d):
        print(f'  {sp}: {len([f for f in os.listdir(d) if f.endswith(".npy")])} files')

## Step 2 — Shared Vocabulary (Paper §3.1)
> Paper uses SentencePiece shared gloss+text vocabulary. We use word-level.

In [ ]:
all_t = []
for sp in ['train','val','test']:
    d = os.path.join(FEATS, sp)
    if not os.path.isdir(d): continue
    for f in os.listdir(d):
        if f.endswith('_label.txt'):
            all_t.extend(tokenize(open(os.path.join(d,f), encoding='utf-8').read()))

vocab = {PAD:0, BOS:1, EOS:2, '<blank>':3}
for i, t in enumerate(sorted(set(all_t)), 4): vocab[t] = i
i2t = {v:k for k,v in vocab.items()}
V = len(vocab)
PI=vocab[PAD]; BI=vocab[BOS]; EI=vocab[EOS]; BK=vocab['<blank>']

json.dump(vocab, open(os.path.join(BASE,'gloss_vocab.json'),'w'), indent=2)
print(f'Shared vocabulary (Paper §3.1): {V} tokens')

## Step 3 — Dataset

In [ ]:
class D2(Dataset):
    def __init__(self, dr, mn=300, tr=False):
        self.s, self.mn, self.tr = [], mn, tr
        for f in sorted(os.listdir(dr)):
            if not f.endswith('.npy'): continue
            lb = os.path.join(dr, f[:-4]+'_label.txt')
            if not os.path.exists(lb): continue
            txt = open(lb, encoding='utf-8').read().strip()
            tks = tokenize(txt)
            if not tks: continue
            gi = [vocab.get(t, PI) for t in tks]
            ti = [BI] + gi + [EI]
            self.s.append((os.path.join(dr,f), gi, ti, txt))
        print(f'  {dr.split(os.sep)[-1]:6s}: {len(self.s)} samples')

    def __len__(self): return len(self.s)

    def __getitem__(self, i):
        p, gi, ti, _ = self.s[i]
        x = np.nan_to_num(np.load(p).astype(np.float32))
        if self.tr:
            if random.random() < 0.5: x += np.random.randn(*x.shape).astype(np.float32)*0.01
            if random.random() < 0.4:
                T=x.shape[0]; r=random.uniform(0.8,1.2)
                idx=np.round(np.linspace(0,T-1,max(int(T*r),4))).astype(int).clip(0,T-1); x=x[idx]
        if x.shape[0] > self.mn:
            idx=np.round(np.linspace(0,x.shape[0]-1,self.mn)).astype(int); x=x[idx]
        return torch.tensor(x,dtype=torch.float32), torch.tensor(gi,dtype=torch.long), torch.tensor(ti,dtype=torch.long)

def collate(b):
    xs,gs,ts=zip(*b)
    xl=torch.tensor([x.shape[0] for x in xs],dtype=torch.long)
    gl=torch.tensor([g.shape[0] for g in gs],dtype=torch.long)
    return (pad_sequence(xs,batch_first=True),
            pad_sequence(gs,batch_first=True,padding_value=PI),
            pad_sequence(ts,batch_first=True,padding_value=PI),xl,gl)

print('Loading datasets...')
tr_ds=D2(os.path.join(FEATS,'train'),tr=True)
vl_ds=D2(os.path.join(FEATS,'val'), tr=False)
assert len(tr_ds)>0, f'No train samples in {FEATS}/train'
assert len(vl_ds)>0, f'No val samples in {FEATS}/val'
tr_ld=DataLoader(tr_ds,batch_size=8,shuffle=True, collate_fn=collate,num_workers=0)
vl_ld=DataLoader(vl_ds,batch_size=8,shuffle=False,collate_fn=collate,num_workers=0)
print(f'Feature dim: 512 | Vocab: {V} | Train: {len(tr_ds)} | Val: {len(vl_ds)}')

## Step 4 — EMF-SLT Model (Paper §2, Fig.2)
| Component | Paper ref | Implementation |
|---|---|---|
| **Vector Quantizer** | §2.1, Eq.1-2 | Gumbel-Softmax, codebook=50 |
| **Fusion Module** | §2.1, Eq.3-5 | `Q=[S;Ŝ']`, `K=V=S`, MultiHead |
| **S2T + G2T decoders** | §2.2, Eq.6 | 2× Transformer decoder |
| **KL mutual learning** | §2.2, Eq.7 | Bidirectional KL divergence |

In [ ]:
class VQ(nn.Module):
    """Vector Quantizer — Paper 2 §2.1, Eq(1)(2). Codebook size=50 (as in paper)."""
    def __init__(self, d=256, cb=50, nq=16, tau=0.1):
        super().__init__()
        self.tau = tau
        self.cb  = nn.Embedding(cb, d)
        self.q   = nn.Parameter(torch.randn(1, nq, d))
        self.as_ = nn.MultiheadAttention(d, 4, batch_first=True, dropout=0.1)
        self.ag  = nn.MultiheadAttention(d, 4, batch_first=True, dropout=0.1)
        self.ps  = nn.Sequential(nn.Linear(d,d), nn.ReLU(), nn.Linear(d,cb))
        self.pg_ = nn.Sequential(nn.Linear(d,d), nn.ReLU(), nn.Linear(d,cb))

    def forward(self, S, G):
        B = S.shape[0]; Q = self.q.expand(B,-1,-1)
        Sh,_ = self.as_(Q, S, S)   # compress sign to N queries
        Gh,_ = self.ag(Q, G, G)    # compress gloss to N queries
        ls = self.ps(Sh); lg = self.pg_(Gh)
        # Gumbel-Softmax — Eq(2)
        p_s = F.gumbel_softmax(ls, tau=self.tau, hard=False)
        p_g = F.gumbel_softmax(lg, tau=self.tau, hard=False)
        # Alignment loss — Eq(1): L_Align = -Σ sg[p_g] log(p_s)
        L_align = -(p_g.detach() * (p_s + 1e-8).log()).sum(-1).mean()
        # Discrete codebook lookup
        Sp = self.cb(p_s.argmax(-1))  # [B, N, D]
        return Sp, Gh, L_align


class Fuse(nn.Module):
    """Fusion Module — Paper 2 Eq(3)-(5): Q=[S;Ŝ'], K=V=S"""
    def __init__(self, d=256):
        super().__init__()
        self.ca = nn.MultiheadAttention(d, 4, batch_first=True, dropout=0.1)
        self.n  = nn.LayerNorm(d)

    def forward(self, S, Sp):
        Q = torch.cat([S, Sp], dim=1)   # Eq(3): Q = [S; Ŝ']
        fM, _ = self.ca(Q, S, S)        # Eq(5): fM = MultiHead(Q, S, S)
        return self.n(fM)


class TDec(nn.Module):
    """Transformer Decoder — 3 layers as in Paper 2"""
    def __init__(self, vs, d=256, p=0.3):
        super().__init__()
        self.emb = nn.Embedding(vs, d, padding_idx=PI)
        self.pe  = nn.Embedding(60, d)
        dl = nn.TransformerDecoderLayer(d, 4, 512, p, batch_first=True)
        self.dec = nn.TransformerDecoder(dl, 3)
        self.out = nn.Linear(d, vs)
        self.dp  = nn.Dropout(p)

    def forward(self, t, mem):
        pos = torch.arange(t.shape[1], device=t.device).unsqueeze(0)
        x   = self.dp(self.emb(t) + self.pe(pos))
        msk = nn.Transformer.generate_square_subsequent_mask(t.shape[1], device=t.device)
        return self.out(self.dec(x, mem, tgt_mask=msk, tgt_key_padding_mask=(t==PI)))


class EMF(nn.Module):
    """Full EMF-SLT — Paper 2 Fig.2"""
    def __init__(self, fdim, vs, d=256):
        super().__init__()
        # Sign encoder (projection + Transformer)
        self.sp = nn.Sequential(nn.Linear(fdim,d), nn.LayerNorm(d), nn.ReLU(), nn.Dropout(0.3))
        el  = nn.TransformerEncoderLayer(d,4,512,0.3,batch_first=True)
        self.senc = nn.TransformerEncoder(el, 3)
        # Gloss encoder (Transformer)
        self.gemb = nn.Embedding(vs, d, padding_idx=PI)
        el2 = nn.TransformerEncoderLayer(d,4,512,0.3,batch_first=True)
        self.genc = nn.TransformerEncoder(el2, 3)
        # VQ + Fusion (Paper 2 core)
        self.vq   = VQ(d, 50, 16)
        self.fuse = Fuse(d)
        # Two decoders: S2T (from fused) + G2T (from gloss) — Paper 2 §2.2
        self.ds = TDec(vs, d)
        self.dg = TDec(vs, d)

    def forward(self, x, g, t):
        S  = self.senc(self.sp(x))                             # sign features
        pm = (g == PI)
        G  = self.genc(self.gemb(g), src_key_padding_mask=pm)  # gloss features
        Sp, _, La = self.vq(S, G)                              # vector quantizer
        fM = self.fuse(S, Sp)                                  # fusion module
        ti = t[:, :-1]
        return self.ds(ti, fM), self.dg(ti, G), La  # S2T logits, G2T logits, L_align


model = EMF(512, V).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'EMF-SLT Parameters: {params:,}')
print('Components: SignEnc + GlossEnc + VQ(cb=50) + Fusion + 2×TDec')

## Step 5 — Training with Multi-Task Loss (Paper 2 Eq.6-8)
`L = L_NLL + α·L_KL + β·L_Align`  where **α=1.0, β=0.5** (from paper §3.1)

Shows **loss ↓ AND token accuracy % ↑** every epoch.

In [ ]:
EP=100; LR=1e-4; PAT=20   # Paper: lr=1e-4, batch=16

ce_c = nn.CrossEntropyLoss(ignore_index=PI, label_smoothing=0.2)  # Paper: label_smooth=0.2
opt  = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9,0.998))  # Paper: Adam β1=0.9 β2=0.998
sch  = torch.optim.lr_scheduler.LambdaLR(opt,
    lambda e: (e+1)/10 if e<10 else 0.5*(1+math.cos(math.pi*(e-10)/max(EP-10,1))))


def token_acc(logits, targets):
    preds = logits.argmax(-1)
    mask  = (targets != PI)
    return ((preds == targets) & mask).sum().item(), mask.sum().item()


def run(ld, tr=True):
    model.train(tr)
    tot, n, correct, total = 0.0, 0, 0, 0
    for xs, gs, ts, xl, gl in ld:
        xs, gs, ts = xs.to(DEVICE), gs.to(DEVICE), ts.to(DEVICE)
        if tr: opt.zero_grad()
        with torch.set_grad_enabled(tr):
            ls2t, lg2t, La = model(xs, gs, ts)
            tgt_out = ts[:, 1:].contiguous()
            # NLL — Eq(6)
            Ls  = ce_c(ls2t.reshape(-1,V), tgt_out.reshape(-1))
            Lg  = ce_c(lg2t.reshape(-1,V), tgt_out.reshape(-1))
            Lnll = Ls + Lg
            # KL mutual learning — Eq(7)
            ps = F.softmax(ls2t.detach(),dim=-1)+1e-8
            pg = F.softmax(lg2t.detach(),dim=-1)+1e-8
            Lkl = (ps*(ps.log()-pg.log())).sum(-1).mean() + (pg*(pg.log()-ps.log())).sum(-1).mean()
            # Total — Eq(8)
            loss = Lnll + ALPHA*Lkl + BETA*La
        if not (torch.isnan(loss) or torch.isinf(loss)):
            if tr:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            tot += loss.item(); n += 1
            c, tc = token_acc(ls2t, tgt_out)
            correct += c; total += tc
    acc = correct / max(total,1) * 100
    return (tot/n, acc) if n>0 else (float('inf'), 0.0)


th, vh, ta, va = [], [], [], []
bv, ni = float('inf'), 0

print(f'EMF-SLT | {EP}ep | L=NLL+{ALPHA}*KL+{BETA}*Align | label_smooth=0.2 | Adam(β1=0.9,β2=0.998)')
print(f'{"Ep":>4}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Loss":>9}  {"Val Acc":>8}')
print('─'*57)

for ep in range(1, EP+1):
    tl, tacc = run(tr_ld, True)
    vl, vacc = run(vl_ld, False)
    sch.step()
    th.append(tl); vh.append(vl); ta.append(tacc); va.append(vacc)
    s = ''
    if vl < bv:
        bv=vl; ni=0
        torch.save({'ep':ep,'m':model.state_dict(),'v':vocab,'l':vl}, CKPT)
        s = '  ✅'
    else: ni += 1
    print(f'Ep {ep:3d}/{EP}  loss={tl:.4f}  acc={tacc:5.1f}%  vl={vl:.4f}  vacc={vacc:5.1f}%{s}')
    if ni >= PAT: print('Early stop'); break

print(f'\nBest val loss: {bv:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5))
fig.suptitle('Paper 2: EMF-SLT Training', color='#58a6ff', fontsize=14)
axes[0].plot(th, color='#3fb950', lw=2, label='Train Loss')
axes[0].plot(vh, color='#f85149', lw=2, label='Val Loss')
axes[0].set_title('Loss ↓ (lower = better)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(ta, color='#3fb950', lw=2, label='Train Acc %')
axes[1].plot(va, color='#f85149', lw=2, label='Val Acc %')
axes[1].set_title('Token Accuracy % ↑ (higher = better)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(BASE,'paper2_training.png'),dpi=150); plt.show()

## Step 6 — Evaluation: BLEU-4 + ROUGE-L (Paper §3.1)

In [ ]:
ckpt = torch.load(CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['m'])
print(f'Loaded: epoch={ckpt["ep"]}  val_loss={ckpt["l"]:.4f}')

def greedy_gen(xf, gids, mx=25):
    model.eval()
    with torch.no_grad():
        x  = torch.tensor(xf).unsqueeze(0).to(DEVICE)
        g  = torch.tensor(gids).unsqueeze(0).to(DEVICE)
        S  = model.senc(model.sp(x))
        G  = model.genc(model.gemb(g))
        Sp,_,_ = model.vq(S, G); fM = model.fuse(S, Sp)
        out = [BI]
        for _ in range(mx):
            t   = torch.tensor([out], dtype=torch.long, device=DEVICE)
            nxt = model.ds(t, fM)[0,-1].argmax().item()
            if nxt == EI: break
            out.append(nxt)
    return [i2t.get(i,'?') for i in out[1:]]

def bleu_n(ref, hyp, n):
    if len(hyp)<n: return 0.0
    rn = set(zip(*[ref[i:] for i in range(n)]))
    hn = [tuple(hyp[i:i+n]) for i in range(len(hyp)-n+1)]
    return sum(1 for g in hn if g in rn)/max(len(hn),1)

def rouge_l(ref, hyp):
    d = np.zeros((len(ref)+1, len(hyp)+1))
    for i in range(1,len(ref)+1):
        for j in range(1,len(hyp)+1):
            d[i][j]=d[i-1][j-1]+1 if ref[i-1]==hyp[j-1] else max(d[i-1][j],d[i][j-1])
    lcs=d[len(ref)][len(hyp)]; p=lcs/max(len(hyp),1); r=lcs/max(len(ref),1)
    return 2*p*r/max(p+r,1e-8)

b1,b2,b3,b4,rl,sa,ex=[],[],[],[],[],[],[]
for np_,gi,ti,lb in vl_ds.s:
    x = np.nan_to_num(np.load(np_).astype(np.float32))
    pred = greedy_gen(x, gi); ref = tokenize(lb)
    b1.append(bleu_n(ref,pred,1)); b2.append(bleu_n(ref,pred,2))
    b3.append(bleu_n(ref,pred,3)); b4.append(bleu_n(ref,pred,4))
    rl.append(rouge_l(ref,pred)); sa.append(1.0 if ref==pred else 0.0)
    if len(ex)<8: ex.append((lb,' '.join(pred) or '<empty>'))

print(f'\n{"═"*55}')
print(f'  BLEU-1       : {np.mean(b1)*100:.2f}%')
print(f'  BLEU-2       : {np.mean(b2)*100:.2f}%')
print(f'  BLEU-3       : {np.mean(b3)*100:.2f}%')
print(f'  BLEU-4       : {np.mean(b4)*100:.2f}%')
print(f'  ROUGE-L      : {np.mean(rl)*100:.2f}%')
print(f'  Sequence Acc : {np.mean(sa)*100:.2f}%  (exact match)')
print(f'{"═"*55}\n')
for r,h in ex:
    print(f'REF: {r}')
    print(f'HYP: {h}\n')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('Paper 2: EMF-SLT Evaluation (Paper §3.1 metrics)', color='#58a6ff', fontsize=14)

bvals = [np.mean(b1)*100, np.mean(b2)*100, np.mean(b3)*100, np.mean(b4)*100]
bars = axes[0].bar(['BLEU-1','BLEU-2','BLEU-3','BLEU-4'], bvals,
    color=['#58a6ff','#3fb950','#ffa657','#f85149'], alpha=0.85, width=0.5)
for bar,v in zip(bars,bvals):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.1f}%',
                 ha='center', color='white', fontsize=11, fontweight='bold')
axes[0].set_title('BLEU Scores'); axes[0].grid(True, alpha=0.3)

mvals = [np.mean(rl)*100, np.mean(sa)*100]
bars2 = axes[1].bar(['ROUGE-L','Sequence Acc'], mvals, color=['#d2a8ff','#79c0ff'], alpha=0.85, width=0.4)
for bar,v in zip(bars2,mvals):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.1f}%',
                 ha='center', color='white', fontsize=13, fontweight='bold')
axes[1].set_title('ROUGE-L + Exact Match'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE,'paper2_eval.png'), dpi=150)
plt.show()